<a href="https://colab.research.google.com/github/smerashanmugan/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [3]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [4]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print('Total revenue:', total_revenue)
print('Total units:', total_units)
print('Rows:', len(df))

Total revenue: 8520.0
Total units: 783
Rows: 400


I first created a new column by multiplying the quantity and price for each order. I then summed the revenue column to identify the total revenue while summing the quanitity column to determine the total number of units that were sold.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [11]:
by_category = df.groupby('category')['revenue'].sum().reset_index()

by_category['share_of_total'] = (
    by_category['revenue'] / df['revenue'].sum() * 100
)

by_category = by_category.sort_values(
    'revenue', ascending=False
)

by_category

,category,revenue,share_of_total
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


I grouped the data by category and then I added up the recenues for each one and then I calculate the percentage for each category of the total revenue. Then I sorted them from highest to lowest so that I could see which ones had the most revenue.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [6]:
vendor_summary = df.groupby('vendor_id').agg(
    avg_order_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
).reset_index()

vendor_summary = vendor_summary.sort_values(
    'avg_order_revenue', ascending=False
)

vendor_summary

,vendor_id,avg_order_revenue,order_count
0,V-01,22.595745,94
3,V-18,21.750000,108
1,V-05,20.580645,93
2,V-10,20.314286,105


I grouped the orders by vendor vedor and then I calculated each vendor's average order revenue and the number of orders. I then sorted the average revenue from highest to lowest so that i could identify the vendor with the highest average and also determine how many orders the average was based on.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [7]:
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()

merch_share = merch_revenue / df['revenue'].sum() * 100

print(f'Merch share of revenue: {merch_share:.1f}%')

Merch share of revenue: 20.8%


I added up the revenue from Merch orders and then I divided it by the total revenue. I then multipled that by 100 to be able to determine what percentage of the overall revenue was coming from the merch.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [15]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})


rows_before = len(df)
revenue_before = df['revenue'].sum()

joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

unmatched = joined.loc[
    joined['vendor_name'].isna(),
    'vendor_id'
].unique()

print('Unmatched vendor:', unmatched)
print('Rows before:', rows_before)
print('Rows after:', len(joined))
print('Revenue before:', revenue_before)
print('Revenue after:', joined['revenue'].sum())


# TODO: merge, validate, and report the unmatched vendor

Unmatched vendor: ['V-18']
Rows before: 400
Rows after: 400
Revenue before: 8520.0
Revenue after: 8520.0


**The unmatched vendor, and what I did about it: I first used a left join to add vendor names to the order and to also keep the existing orders. The many_to_one validation was used to check if the vendor ID matched at most one vendor name. I also made sure to check that the row count and total revenue stayed the same after the merge. I then kept the unmatched orders the v-18 as unmatched vendors to prevent the data from getting lost.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [9]:
pivot = pd.pivot_table(
    df,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Total,972.0,3274.5,1263.0,661.5,6171.0


For this question, I created a pivot table that is able to sum the revenue for each vendor by category where the vendors are in the rows and the categories in the columns. I then added the row and column totals so that i t would be easier to be able to compare the revenue across vendors and categories.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [16]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

1. I would tell the vendors that for the next game that they would need to focus on food because it generated the largest revenue at 4,293 dollars which is about the 50 % of the total revenue. The second highest was the merch at 1,771.50 dollars which is alomst 21 % of the total revenue. These results suggest that vendores should make sure that they have enough food available du to the high demand for it while also makin gsure that merch is available too.
2. I think that question 5 had the least trustworthy answer as v-18 did not have a vendor name in the table. I kept those specific orders are "unknown vendor" so that the date would not be lost but also this makes the report incomplete.